# Guerilla Search — Context-Aware Website Analysis

Guerilla Search is an LLM-powered website analysis tool that goes beyond traditional summarization.

Instead of simply condensing webpage content, the system explores the broader context behind what a website presents. Given a URL, it extracts website content and generates a structured analysis that distinguishes between explicit messaging and inferred context.

The project highlights:

* **Official Story** — what the website explicitly communicates
* **Hidden Context** — possible assumptions, incentives, and framing
* **Missing Voices** — perspectives that may not be represented
* **Public Reaction** — hypothetical viewpoints from different audiences
* **Guerilla Verdict** — a concise, witty final observation

The system is designed to encourage critical reading of web content while separating facts from inference and avoiding unsupported claims.

This project demonstrates how Large Language Models can move beyond summarization into contextual interpretation and perspective-aware analysis.

In [8]:
import os
import sys
from dotenv import load_dotenv
sys.path.append("../..")
from week1.scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

In [9]:
load_dotenv(override=True)

api_key = os.getenv("GOOGLE_API_KEY")

if not api_key:
    raise ValueError("GOOGLE_API_KEY not found in .env")

if api_key.strip() != api_key:
    raise ValueError("GOOGLE_API_KEY contains spaces")

print("GOOGLE_API_KEY loaded successfully")

GOOGLE_API_KEY loaded successfully


In [10]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

client = OpenAI(base_url=GEMINI_BASE_URL, api_key=api_key)

In [11]:
system_prompt = """
You are Guerilla Search.

Your goal is NOT to summarize websites.

Your goal is to reveal hidden context and societal undercurrents.

For every website:

1. Explain the official message.
2. Infer possible motivations.
3. Identify missing perspectives.
4. Mention who benefits.
5. Mention groups that may disagree.
6. Point out assumptions or marketing language.
7. Add mild humor and irony.

Never invent facts.
Separate:
- Facts
- Inferences
- Speculation

Respond in markdown.
"""

In [12]:
user_prompt_prefix = """
Analyze this website.

Website Content:
"""

In [13]:
def messages_for(context):

    return [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt_prefix + context
        }
    ]

In [14]:
def summarize(url):

    print(f"Fetching: {url}")

    website = fetch_website_contents(url)

    context = f"""
WEBSITE CONTENT:

{website}

Instructions:

- Official Story → facts only
- Hidden Context → cautious inference
- Missing Voices → only if noticeable
- Public Reaction → hypothetical
- Never invent events
"""

    response = client.chat.completions.create(
        model="gemini-2.5-flash-lite",
        messages=messages_for(context)
    )

    return response.choices[0].message.content

In [15]:
def display_summary(url):

    try:

        result = summarize(url)

        display(Markdown(result))

    except Exception as e:

        print("Error:")
        print(e)

In [17]:
display_summary("https://wikipedia.org")

Fetching: https://wikipedia.org


Here's a Guerilla Search analysis of the provided Wikipedia content:

### Official Message

Wikipedia presents itself as "The Free Encyclopedia." The provided text highlights the vast number of articles available in numerous languages, with a specific emphasis on the top languages like English, Japanese, German, Russian, French, Spanish, Chinese, Italian, Polish, and Portuguese, each boasting over a million articles. It also lists languages with over 100,000 and 10,000 articles, showcasing its global reach and multilingual accessibility. The implicit message is that Wikipedia is a comprehensive, easily accessible, and globally recognized source of information, freely available to anyone.

### Possible Motivations

The primary motivation behind showcasing these numbers is to reinforce Wikipedia's brand as a dominant and authoritative global knowledge repository. It aims to:

*   **Attract and Retain Users:** By demonstrating its sheer volume and breadth, Wikipedia encourages users to rely on it as their first stop for information.
*   **Attract Contributors:** A large, active community is a sign of a healthy project. Showcasing article counts might inspire potential editors to join and contribute, particularly in languages that might be underrepresented.
*   **Seek Donations:** While not explicitly stated, large organizations often highlight their achievements and scale to solicit financial support. A successful, globally recognized platform is a more compelling recipient of donations.
*   **Maintain Relevance:** In an age of rapidly evolving information platforms, showing continuous growth and broad language support helps Wikipedia maintain its position as a key player.

### Missing Perspectives

*   **Quality vs. Quantity:** The article count metric doesn't speak to the accuracy, depth, or neutrality of the content in each language. A million articles in one language might be vastly different in scope and reliability than a million in another.
*   **Editor Demographics:** Who are these editors? Are they diverse in terms of geography, expertise, and demographics, or is there a concentration of editors from specific regions or backgrounds? This impacts the perspectives reflected in the articles.
*   **Content Gaps:** While impressive, the numbers don't reveal what *isn't* covered. Are there entire fields of knowledge or specific cultural perspectives that are significantly underrepresented across all languages?
*   **Commercial Influence:** The text doesn't touch on how commercial interests, advertising (or lack thereof), and platform policies might shape the content.

### Who Benefits?

*   **The General Public:** Access to a vast, free encyclopedia in their native language is a significant benefit.
*   **Students and Researchers:** Wikipedia serves as a starting point and a quick reference, saving time and effort.
*   **The Wikimedia Foundation:** As the non-profit organization behind Wikipedia, they benefit from the platform's perceived authority and reach, which aids in fundraising and advocacy.
*   **Editors and Contributors:** They benefit from the platform and the community, contributing to their own knowledge and skill development.

### Groups That May Disagree

*   **Traditional Academic Institutions/Publishers:** They might argue that Wikipedia, despite its improvements, can never fully replace peer-reviewed journals or scholarly books for in-depth, authoritative research.
*   **Niche Knowledge Keepers:** Individuals or groups whose specific expertise or cultural knowledge is not adequately represented on Wikipedia might feel its "comprehensiveness" is overstated.
*   **Censorship Advocates (Hypothetical):** While Wikipedia aims for neutrality, its content can be a point of contention for groups who believe certain information should be suppressed or controlled.
*   **Those Critical of Online Information Authority:** Skeptics of crowdsourced information might question the inherent trustworthiness of a platform built by volunteers.

### Assumptions or Marketing Language

*   **"The Free Encyclopedia":** This is a powerful marketing slogan. While the content is free to *access*, the *creation* and *maintenance* are not without cost (server infrastructure, staff, etc.), which are funded by donations. The "free" aspect primarily refers to the licensing and accessibility.
*   **Emphasis on Numbers:** The sheer volume of articles is presented as an indicator of success and quality. This is a common marketing tactic to impress and demonstrate scale. It assumes that more articles automatically equate to a better or more valuable resource.
*   **Global Reach as Inherent Goodness:** The multilingual aspect is framed as universally positive, implying that it inherently promotes understanding and knowledge sharing, without acknowledging potential language-specific biases or challenges.

---

So, Wikipedia is basically saying, "Look how big we are! In *so many languages*! Aren't we amazing?" It's a digital Tower of Babel, but instead of confusion, they promise knowledge. One wonders if the editors in Esperanto feel a bit lonely in their corner, despite having 10,000+ articles. It's a marvel of collective effort, but as always, the devil (and the nuance) is in the details not advertised on the homepage.